In [0]:
from pyspark.sql.functions import *

In [0]:
# Lire la table reviews Bronze
df_reviews = spark.table("`E-commerce`.bronze.reviews")

In [0]:
# Afficher les types des colonnes
df_reviews.printSchema()

In [0]:
# Supprimer les lignes identiques
reviews_silver = df_reviews.dropDuplicates()

In [0]:
# Garder un seul avis par review_id
reviews_silver = reviews_silver.dropDuplicates(["review_id"])

In [0]:
# Supprimer les avis sans review_id
reviews_silver = reviews_silver.filter(
    col("review_id").isNotNull()
)

In [0]:
# Remplacer les ratings invalides par NULL
reviews_silver = reviews_silver.withColumn(
    "rating",
    when(
        (col("rating") >= 1) & (col("rating") <= 5),
        col("rating")
    ).otherwise(None)
)

In [0]:
# Remplacer les votes negatifs par NULL
reviews_silver = reviews_silver.withColumn(
    "helpful_votes",
    when(col("helpful_votes") < 0, None)
    .otherwise(col("helpful_votes"))
)

In [0]:
# Garder les avis avec un client valide
reviews_silver = reviews_silver.join(
    spark.table("`E-commerce`.silver.customers").select("customer_id"),
    "customer_id",
    "inner"
)

In [0]:
# Garder les avis avec un produit valide
reviews_silver = reviews_silver.join(
    spark.table("`E-commerce`.silver.products").select("product_id"),
    "product_id",
    "inner"
)

In [0]:
# Comparer le nombre d'avis
print("Bronze :", df_reviews.count())
print("Silver :", reviews_silver.count())

In [0]:
# Enregistrer reviews dans Silver
reviews_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("`E-commerce`.silver.reviews")

In [0]:
# Afficher les tables Silver
spark.sql("SHOW TABLES IN `E-commerce`.silver").show()